In [373]:
import pandas as pd
import numpy as np

In [374]:
data = pd.read_csv("genre_added.csv", index_col=["tmdb_id"])

data.head()

,imdb_id,title,release_date,language,rating_tmdb,rating_imdb,votes_imdb,revenue,director,cast,genres,runtimeMinutes
tmdb_id,,,,,,,,,,,,
487288,NaN,A Mercedes for Ashish,2006-01-01,hi,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
442680,tt4944494,Shaitani Dracula,2006-01-01,hi,0.0,4.6,21.0,0.0,Harinam Singh,"Harinam Singh, Birbal, Ramesh Goyal, Manmauji,...",Horror,79.0
1436302,NaN,Intoxication of the Body,2006-01-01,hi,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1357257,NaN,The Heartbeat,2006-01-01,hi,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
446321,NaN,Free Entry,2006-01-01,hi,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [375]:
print(f"Data Shape:{data.shape}")
print(f"NA count: \n{data.isna().sum()}")

Data Shape:(9079, 12)
NA count: 
imdb_id           3120
title                0
release_date         0
language             0
rating_tmdb          0
rating_imdb       3479
votes_imdb        3479
revenue           2115
director          2496
cast              2499
genres            3569
runtimeMinutes    4000
dtype: int64


In [376]:
data = data.dropna(subset=["votes_imdb"])
data[["director", "cast", "genres"]] = data[["director", "cast", "genres"]].fillna(
    "Unknown"
)
data["runtimeMinutes"] = data["runtimeMinutes"].fillna(data["runtimeMinutes"].mean()).astype(int)

In [377]:
train_data=data[data['revenue']!=0]
test_data=data[data['revenue']==0]
print(f"Train Data Shape:{train_data.shape}")

print(f"Test Data Shape:{test_data.shape}")

Train Data Shape:(488, 12)
Test Data Shape:(5112, 12)


In [378]:
train_X=train_data.loc[:,['language','rating_tmdb','votes_imdb']]
train_Y=train_data.loc[:,['revenue']]
test_X=test_data.loc[:,['language','rating_tmdb','votes_imdb']]
test_Y=test_data.loc[:,['revenue']]


In [379]:

print(f"Train X Shape:{train_X.shape}")
print(f"DataTypes:{train_X.dtypes}")

Train X Shape:(488, 3)
DataTypes:language        object
rating_tmdb    float64
votes_imdb     float64
dtype: object


In [380]:
train_Y.head()

,revenue
tmdb_id,
69412,500000.0
7913,11502151.0
81060,5800000.0
79667,2800000.0
63818,3700000.0


In [381]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(train_X[['language']])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(['language']),
    index=train_X.index
)

train_X = pd.concat([train_X, encoded_df], axis=1)
train_X.drop('language', axis=1, inplace=True)



In [382]:
encoded_test=encoder.transform(test_X[['language']])
encoded_df = pd.DataFrame(
    encoded_test,
    columns=encoder.get_feature_names_out(['language']),
    index=test_X.index
)
test_X = pd.concat([test_X, encoded_df], axis=1)
test_X.drop('language', axis=1, inplace=True)


In [383]:
from sklearn.linear_model import LinearRegression
lr=LinearRegression()
lr.fit(train_X,train_Y)

LinearRegression()

In [384]:
predictions=lr.predict(test_X)
predictions=predictions.astype(int)
predictions

array([[ 5253714],
       [ 5294010],
       [ 2074698],
       ...,
       [ 5456859],
       [ 8039355],
       [13635752]], shape=(5112, 1))

In [385]:
predictions = pd.DataFrame(
    predictions,
    columns=['revenue'],
    index=test_X.index,
)
predictions.head()

,revenue
tmdb_id,
442680,5253714
272222,5294010
486382,2074698
1358349,5248606
272210,5253147


In [386]:
data.loc[predictions.index,'revenue']=predictions
print(f"Data Shape:{data.shape}")

Data Shape:(5600, 12)


In [387]:
print(f"NA count: \n{data.isna().sum()}")

NA count: 
imdb_id           0
title             0
release_date      0
language          0
rating_tmdb       0
rating_imdb       0
votes_imdb        0
revenue           0
director          0
cast              0
genres            0
runtimeMinutes    0
dtype: int64


In [388]:
data.to_csv('add_revenue.csv')